# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

**All entities in this notebook (record sets, fields, columns) are referenced by their `@id` values as per the Croissant standard.**

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. The dataset metadata describes all record sets, fields, and files available.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
from pprint import pprint

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print dataset title and description
print(f"{metadata.name}: {metadata.description}")
# Show dataset version and license
print(f"\nVersion: {metadata.version}\nLicense: {metadata.license}")
print(f"\nPublished on: {metadata.datePublished}")

## 2. Data Overview
Let's review the available record sets, their fields, and their `@id` values. This will help us reference the correct entities in downstream steps.

`mlcroissant` allows us to programmatically inspect all record sets and fields (columns) in the schema. Each is uniquely identified by its `@id` field.

In [ ]:
# List all record sets defined in the schema
print("Available record sets (by @id):\n")
for record_set in metadata.recordSets:
    print(f" • {record_set['@id']} : {record_set.get('name', '[no name]')}")

# For each record set, show its fields
print("\nFields (by @id) for each record set:\n")
for record_set in metadata.recordSets:
    print(f"Record Set: {record_set.get('name', '[no name]')} (@id={record_set['@id']})")
    fields = record_set.get('fields', []) or record_set.get('columns', [])  # 'fields' or 'columns', schema-dependent
    for field in fields:
        field_id = field.get('@id')
        field_name = field.get('name', '[no field name]')
        print(f"    - {field_id} : {field_name}")
    print("")
# Save first record_set id for next sections
primary_record_set_id = metadata.recordSets[0]['@id'] if len(metadata.recordSets) > 0 else None

## 3. Data Extraction

Let's load tabular data from a specific record set into a [Pandas](https://pandas.pydata.org/) DataFrame for analysis. We reference record set and field `@id`s as found in the schema above.

We'll demonstrate with the primary record set (the first available), but you can repeat for any using the correct `@id`.

In [ ]:
# List all record set @id's
record_set_ids = [rs['@id'] for rs in metadata.recordSets]

dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for record set {record_set_id}")
    except Exception as e:
        print(f"Could not load records for record set {record_set_id}: {e}")

# For exploration, use the first loaded record set (primary_record_set_id):
if primary_record_set_id and primary_record_set_id in dataframes:
    print("\nColumns in the primary record set DataFrame:")
    print(dataframes[primary_record_set_id].columns.tolist())
    display(dataframes[primary_record_set_id].head())
else:
    print("No primary record set data loaded.")

## 4. Exploratory Data Analysis (EDA)

Let's perform basic data processing:
- Filter records on a numeric field (by `@id`)
- Normalize a numeric field
- Optionally group by a categorical field

_Update the field choices to fit your selected record set. Try fields like "coverage", "peptide_counts" or similar (check the printed columns above)._

In [ ]:
# EDIT THESE IDS for your dataset (see above terminal output)

# Example field @ids (replace with actual from step 2 output!)
# e.g. 'cr:peptideCount', 'cr:coverage', etc.
numeric_field_id = None  # e.g. 'cr:coverage' or similar
group_field_id = None    # e.g. 'cr:accession' or similar

# Use the primary record set DataFrame
df = dataframes.get(primary_record_set_id)

if df is not None and numeric_field_id in df.columns:
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a field if desired
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"\nGrouped data by {group_field_id}:")
        display(grouped_df.head())
    else:
        print("Set `group_field_id` to a valid column to group data.")
else:
    print("Please set `numeric_field_id` to a valid numeric column from the above list.")

## 5. Visualization

Visualize data distributions or relationships using fields by `@id`. You may use [`matplotlib`](https://matplotlib.org/) or [`seaborn`](https://seaborn.pydata.org/). Try editing the code below to reference another field by its `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: Distribution of the chosen numeric field
if df is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()
else:
    print("Set `numeric_field_id` above to a valid field to visualize.")

## 6. Conclusion

- We loaded the FAIR^2 dataset via its Croissant schema and explored its metadata and structure using `mlcroissant`.
- Data was referenced, loaded, filtered, and visualized using `@id` fields as required for FAIR reproducibility.
- For deeper analysis, use the list of record sets and field `@id`s to select other slices of data.

For more, see: [mlcroissant GitHub](https://github.com/mlcommons/croissant) or [FAIR^2 dataset on sen.science](https://sen.science/doi/10.71728/senscience.vq4a-28xa/).